# 02 Zero-shot baseline and its reports

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CoralsINPA/RedSeaCoralIdentifier/blob/main/notebooks/02_zero_shot.ipynb)

Run it in Colab on a **GPU runtime** (Runtime > Change runtime type > T4). Run the first cell, then the others in order.

**What it does.** (a) Labels every kept crop of the run with the fine-tuned BioCLIP (`taxonomy_v1` prompts, the 20 hard-coral genera, masked crops) and saves the predictions and the image embeddings. This is the new baseline. (b) Reports on it: the expert-set evaluation, the genus mix against the survey counts, and both broken down by photo orientation.

**Safe to run and re-run.** It reads the shared Drive and only creates new files. It never overwrites one: if the baseline files already exist they are loaded, and each report goes into a new folder.

| Step | What it does | Time | Reads | Writes |
|---|---|---|---|---|
| Setup | mounts Drive, updates the clone, installs `coralid[clip,viz]` | 3-5 min | GitHub | nothing on Drive |
| Config | loads the run config, checks the inputs | seconds | `<run>/manifest/manifest.parquet`, the expert set, the survey counts | nothing |
| (a) Baseline | one pass over every `keep == True` row of the manifest (16,080 crops, 1,714 photos) | **about 25 min** on a T4 (Check B ran at 0.8 s per image), **resumable** | photos, mask JSONs, the model from Hugging Face | `<run>/predictions/finetuned-bioclip__taxonomy_v1__hard20__all.parquet` and `<run>/features/finetuned-bioclip__masked.npz`, each with a `.meta.json`; progress in `<run>/predictions/parts/` |
| (b) Reports | expert-set evaluation, genus distribution against `ground_truth_genus_counts.csv` (the 20 genera only), orientation breakdown | seconds | the baseline, the manifest, `<run>/assaf_expert_test_set.csv` | `<run>/reports/baseline_<name>_<date>/` (a new folder) |

If Colab disconnects during the baseline, reconnect, run **Setup** and **Config** again, then run the baseline cell again: it continues where it stopped, and any image that failed is retried. The two result files are written only when every image is done.

The same job runs from a terminal as `python -m coralid zero-shot --config configs/iou60_sta55_pts32.yaml`.

**Portrait photos (221 of 1,714).** The legacy code cut the wrong region on them, and Assaf's expert labels were given on crops cut by the legacy code. So the expert evaluation is shown for expert rows on **landscape** photos (the fair comparison with the legacy result) and on **portrait** photos (informational: his label may describe a different-looking crop) separately.

**What to paste back to the next session:** the printed tables of the report cells (expert evaluation, paired table, orientation summary).

In [ ]:
import os
import subprocess
import sys

# ================================ SETTINGS: the only place to edit ================================
DATA_ROOT = "/content/drive/MyDrive/colab notebooks/CoralsMasking"  # the Drive folder CoralsMasking
BRANCH = "main"
REPO_URL = "https://github.com/CoralsINPA/RedSeaCoralIdentifier"
REPO_DIR = "/content/RedSeaCoralIdentifier"
EXTRAS = "clip,viz"
# ==================================================================================================

# The package cannot install itself before it exists. A clone left by an earlier session of this
# runtime is brought to the branch head BEFORE anything is imported from it.
if os.path.isdir(f"{REPO_DIR}/.git"):
    steps = [
        ["git", "-C", REPO_DIR, "fetch", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "checkout", BRANCH],
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "submodule", "update", "--init"],
    ]
else:
    steps = [["git", "clone", "--recurse-submodules", "--branch", BRANCH, REPO_URL, REPO_DIR]]
for step in steps:
    print("$", " ".join(step))
    subprocess.run(step, check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # enough to import coralid.colab (standard library only)

from coralid.colab import setup  # noqa: E402

env = setup(
    EXTRAS, data_root=DATA_ROOT, branch=BRANCH, repo_url=REPO_URL, repo_dir=REPO_DIR, update=False
)

In [ ]:
import logging
import time

import torch

from coralid.baseline import baseline_paths
from coralid.config import load_config

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("coralid").setLevel(logging.INFO)

cfg = load_config(f"{REPO_DIR}/configs/iou60_sta55_pts32.yaml")  # reads CORALID_DATA_ROOT
zs = cfg.zero_shot
paths = baseline_paths(cfg)
print(
    f"run {cfg.run_name} | model {zs.model} | prompts {zs.prompt_set} ({zs.prompt_format}) "
    f"| labels {zs.label_set} | crops {cfg.crops.mode}"
)
for label, path in {
    "photos": cfg.images_dir,
    "mask JSONs": cfg.json_dir,
    "manifest": cfg.manifest_path,
    "expert set": cfg.run_dir / "assaf_expert_test_set.csv",
    "survey counts": cfg.ground_truth_path,
    "predictions": paths.predictions,
    "embeddings": paths.features,
}.items():
    print(f"  {label:13} {'ok' if path.exists() else 'not there yet':13} {path}")
if not cfg.manifest_path.exists():
    raise FileNotFoundError(
        "The manifest is missing. It comes from Check C of notebooks/parity_check.ipynb."
    )
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (far too slow)")

## (a) The baseline job (about 25 minutes, GPU, resumable)

`run_baseline` classifies every kept crop and saves, as new files only, the predictions (the 20 genus probabilities, `predicted_genus`, `confidence`, `runner_up` and `margin`) and the L2-normalized image embeddings, each with a `.meta.json` that records the config, git commit, label set, prompts and manifest. If both files already exist it loads them and computes nothing.

In [ ]:
from coralid.baseline import run_baseline

t0 = time.perf_counter()
baseline = run_baseline(cfg, progress=True)
photos = baseline.predictions["image_name"].nunique()
print(
    f"\n{baseline.status} in {time.perf_counter() - t0:.0f} s: "
    f"{len(baseline.predictions)} crops from {photos} photos"
)
print("predictions:", baseline.paths.predictions)
print("embeddings: ", baseline.paths.features, baseline.features.shape)
baseline.predictions[
    ["image_name", "coral_id", "predicted_genus", "confidence", "runner_up", "margin"]
].head()

## (b) Reports on the baseline

The expert set is evaluated three ways: all rows, expert rows on landscape photos, and expert rows on portrait photos. The predictions stored in `assaf_expert_test_set.csv` (the legacy ones) are scored on the same rows and paired with the new ones. **Read the portrait rows with care:** Assaf judged crops cut by the legacy code, which were mis-cut on portrait photos, so a change there is not an improvement or a regression until he has looked at the new crops.

In [ ]:
from coralid.evaluation import load_expert_set, load_legacy_predictions
from coralid.manifest import read_manifest
from coralid.reports import baseline_report

manifest = read_manifest(cfg.manifest_path)
EXPERT_CSV = cfg.run_dir / "assaf_expert_test_set.csv"
expert = load_expert_set(EXPERT_CSV, zs.label_set)
legacy_predictions = load_legacy_predictions(EXPERT_CSV)  # what the CSV itself says
report = baseline_report(
    baseline.predictions,
    manifest,
    expert,
    legacy_predictions,
    cfg.ground_truth_path,
    zs.label_set,
)

print("Expert set, the new baseline, in the label set (Wilson 95% interval):")
for group, evaluation in report.expert_new.items():
    acc = evaluation.accuracy_in_set
    low, high = acc["ci95"] or (float("nan"), float("nan"))
    share = acc["accuracy"] if acc["accuracy"] is not None else float("nan")
    cov = evaluation.coverage
    print(
        f"  {group:9} expert rows {cov['n_expert']:3} | with a prediction {cov['n_joined']:3} "
        f"| correct {acc['correct']}/{acc['total']} = {share:.3f} ({low:.3f}-{high:.3f}) "
        f"| {evaluation.status_counts}"
    )
print("\nNew against legacy on the same rows (paired):")
display(report.expert_paired.round(3))
print("Per-genus recall and precision, landscape rows (the fair comparison):")
display(report.expert_new["landscape"].per_genus.round(3))
print("Top confusions, landscape rows:")
display(report.expert_new["landscape"].top_confusions)

In [ ]:
import matplotlib.pyplot as plt

from coralid.reports import new_report_folder, save_figure

figures = new_report_folder(cfg.reports_dir, "figures_02")  # a new folder for this run's pictures

dist = report.distribution["all"]
print("Predicted genus mix against the survey counts (the 20 genera only, shares within the 20):")
display(dist.round(4))
print("survey genera left out:", dist.attrs["survey_excluded"])

x = list(range(len(dist)))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar([i - 0.2 for i in x], dist["predicted_share"], width=0.4, label="baseline predictions")
ax.bar([i + 0.2 for i in x], dist["survey_share"], width=0.4, label="survey counts")
ax.set_xticks(x, list(dist.index), rotation=60, ha="right")
ax.set_ylabel("share of the 20 genera")
ax.set_title("Baseline predictions against the survey")
ax.legend()
plt.tight_layout()
print("Saved:", save_figure(fig, figures, "genus_mix_vs_survey"))
plt.show()

In [ ]:
mix = report.genus_by_orientation
print("Predictions and genus mix by photo orientation:")
display(report.orientation_summary.round(3))

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar([i - 0.2 for i in x], mix["landscape_share"], width=0.4, label="landscape photos")
ax.bar([i + 0.2 for i in x], mix["portrait_share"], width=0.4, label="portrait photos")
ax.set_xticks(x, list(mix.index), rotation=60, ha="right")
ax.set_ylabel("share of the predictions")
ax.set_title("Genus mix by orientation")
ax.legend()
plt.tight_layout()
print("Saved:", save_figure(fig, figures, "genus_mix_by_orientation"))
plt.show()

In [ ]:
from coralid.viz import confusion_heatmap

print("Expert set, landscape rows: true genus (rows) against predicted genus (columns).")
print("The diagonal is right; every cell off it is a mix-up.")
heatmap = confusion_heatmap(
    report.expert_new["landscape"].confusion, title="Expert set, landscape rows"
)
display(heatmap)
print("Saved:", save_figure(heatmap, figures, "expert_confusion_landscape"))

### See the predictions with your eyes

Random crops of each predicted genus, cut exactly as the model saw them (outside the mask painted black). The caption is the predicted genus and its confidence: green is 0.85 or more, orange 0.5 to 0.85, red below 0.5. Run it again for a new random sample (change `SEED`). Reading the photos from Drive takes about a minute.

In [ ]:
from coralid.masks import AnnotationStore
from coralid.viz import cut_crops

# ================================ SETTINGS ================================
PER_GENUS = 3  # crops shown per predicted genus
SEED = 0  # change it for another random sample
# ==========================================================================
sample = (
    baseline.predictions.sample(frac=1, random_state=SEED)
    .groupby("predicted_genus")
    .head(PER_GENUS)
    .sort_values(["predicted_genus", "confidence"], ascending=[True, False])
)
thumbs = cut_crops(sample, AnnotationStore(cfg.json_dir), cfg.images_dir, cfg.crops, max_side=256)
colors = ["green" if c >= 0.85 else "orange" if c >= 0.5 else "red" for c in sample["confidence"]]
n_cols = 2 * PER_GENUS
n_rows = -(-len(sample) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.2 * n_rows), squeeze=False)
for i, ax in enumerate(axes.ravel()):
    ax.axis("off")
    if i < len(thumbs) and thumbs[i] is not None:
        ax.imshow(thumbs[i])
        row = sample.iloc[i]
        ax.set_title(f"{row.predicted_genus} {row.confidence:.2f}", color=colors[i], fontsize=10)
fig.suptitle(f"{PER_GENUS} random crops per predicted genus (seed {SEED})")
plt.tight_layout()
print("Saved:", save_figure(fig, figures, f"crops_per_genus_seed{SEED}"))
plt.show()

In [ ]:
from coralid.reports import write_baseline_report

folder = write_baseline_report(report, cfg.reports_dir, paths.name)
print("Saved (a new folder, nothing was overwritten):", folder)
for file in sorted(folder.iterdir()):
    print("  ", file.name)

## Next

- Paste the printed tables of the report cells into the next session.
- Open `03_expert_review.ipynb`: the portrait-impact report and the Millepora blind review build on the files this notebook wrote.
- Open `04_supervised_probe.ipynb` once labels exist. (Looking at the baseline's embeddings with maps and clusters is a separate direction, outside the pipeline: `notebooks/explore/embeddings.ipynb`.)